In [ ]:
#Auto-reload edited modules (e.g. src/utils) without restarting the kernel
%load_ext autoreload
%autoreload 2

In [ ]:
#Install dependencies (only the ones that are missing)
from utils import ensure_installed

ensure_installed(["anthropic", "python-dotenv"])

In [ ]:
#Create the Claude client (loads .env)
from utils import ClaudeClient

claude = ClaudeClient( max_tokens=100)
claude.model

In [ ]:
prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

In [ ]:
import json

# Haiku does not support assistant-message prefill ("```json" + stop sequence),
# so ask for structured output with a JSON schema instead.
schema = {
    "type": "object",
    "properties": {
        "tasks": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {"task": {"type": "string"}},
                "required": ["task"],
                "additionalProperties": False,
            },
        }
    },
    "required": ["tasks"],
    "additionalProperties": False,
}

messages = []
claude.add_user_message(messages, prompt)
text = claude.chat(
    messages,
    max_tokens=1000,
    output_config={"format": {"type": "json_schema", "schema": schema}},
)
dataset = json.loads(text)["tasks"]
dataset

In [ ]:
from pathlib import Path

# Project root = first parent folder that contains pyproject.toml
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())

output_path = PROJECT_ROOT / "data" / "test-data.json"
output_path.parent.mkdir(parents=True, exist_ok=True)   # create the folder if it doesn't exist

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(dataset, f, indent=2)

print(f"Saved {len(dataset)} items to {output_path}")

In [ ]:
def run_prompt(test_case):
    prompt = f"""
    Please solve the following task:
    {test_case["task"]}
    """
    messages = []
    claude.add_user_message(messages, prompt)
    output = claude.chat(
        messages,
        max_tokens=1000,
    )
    return output


In [ ]:
def run_test_case(test_case):
    output = run_prompt(test_case)

    score = 10

    return {
        "score": score,
        "output": output,
        "test_case": test_case
    }

In [ ]:
def run_all_test_cases(test_cases):
    results = []
    for test_case in test_cases:
        result = run_test_case(test_case)
        results.append(result)
    return results

In [ ]:
with open(output_path, "r", encoding="utf-8") as f:
    dataset = json.load(f)

results = run_all_test_cases(dataset)   


In [ ]:
print(json.dumps(results, indent=2))